# FTP Honeypot JSON Log Pipeline and Command Analysis

---

## Dataset Overview
* **Source:** Raw JSON event logs from T-Pot / Dionaea honeypots (`RawHoneynetEventsBatch-*.json`).
* **Target Protocol:** FTP (Port 21, `ftpd` protocol events).
* **Scope:** Parsing raw JSON structures, extracting FTP command execution sequences, and building structured threat attributes.

---

## Processing Steps
1. **Extraction:** Decompressed raw JSON events and isolated specific Dionaea FTP logs.
2. **Data Cleaning:** Removed constant, empty, and non-informative headers to reduce dimensionality.
3. **GeoIP Rationalization:** Standardized country codes and eliminated redundant geographic fields.
4. **Feature Engineering:** Extracted standard FTP commands (`USER`, `PASS`, `AUTH`, `EPSV`, `QUIT`), created command frequency metrics (`command_count`), and generated binary authentication flags (`has_USER`, `has_PASS`, `is_anonymous`).

---

## Final Output
* **File:** `ftp_clean_final.csv` – A structured dataset optimized for analyzing FTP threat command behaviors.

In [ ]:
import json
import glob
import os

files = sorted(glob.glob("/content/RawHoneynetEventsBatch-*.json"))

print("Number of files:", len(files))
print("=" * 50)

for file in files:
    size_mb = os.path.getsize(file) / (1024 * 1024)

    print(f"File: {os.path.basename(file)}")
    print(f"Size: {size_mb:.2f} MB")

    with open(file, "r", encoding="utf-8") as f:
        first_line = f.readline()

    try:
        event = json.loads(first_line)

        print("First event type:", type(event))
        print("Top-level keys:")
        print(list(event.keys()))

        print("\n_source keys:")
        print(list(event.get("_source", {}).keys()))

    except Exception as e:
        print("ERROR:", e)

    print("=" * 50)

Number of files: 3
File: RawHoneynetEventsBatch-0.json
Size: 94.00 MB
First event type: <class 'dict'>
Top-level keys:
['_index', '_id', '_score', '_source', 'fields', 'sort']

_source keys:
['@version', 't-pot_hostname', 't-pot_ip_ext', 'geoip_ext', 't-pot_ip_int', 'timestamp', 'message', 'eventid', 'sensor', '@timestamp', 'host', 'path', 'type', 'duration', 'headers', 'src_ip', 'geoip', 'session']
File: RawHoneynetEventsBatch-1.json
Size: 96.00 MB
First event type: <class 'dict'>
Top-level keys:
['_index', '_id', '_score', '_ignored', '_source', 'fields', 'ignored_field_values', 'sort']

_source keys:
['app_version', 'transport_type', '@timestamp', 'sip_message', 'src_ip', 'headers', 'geoip_ext', 'dest_ip', 'host', 'dest_port', 'event_uuid', 'sip_method', 'geoip', 'type', 'app_name', 'called_number', 'collected_method', 'sip_user_agent', 't-pot_ip_int', 'created_by_node_id', '@version', 'path', 't-pot_hostname', 't-pot_ip_ext']
File: RawHoneynetEventsBatch-2.json
Size: 99.00 MB
First

In [ ]:
import json
import glob

files = sorted(glob.glob("/content/RawHoneynetEventsBatch-*.json"))

ftp_events = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                event = json.loads(line)
                src = event.get("_source", {})

                # Precise FTP indicators
                is_ftp = (
                    src.get("type") == "Dionaea"
                    and (
                        src.get("dest_port") == 21
                        or src.get("connection", {}).get("protocol") == "ftpd"
                        or "ftp" in src
                    )
                )

                if is_ftp:
                    ftp_events.append(src)

            except json.JSONDecodeError:
                continue

print("Total precise FTP events:", len(ftp_events))

print("\nFTP types:")
from collections import Counter
print(Counter(e.get("type") for e in ftp_events))

print("\nFTP destination ports:")
print(Counter(e.get("dest_port") for e in ftp_events).most_common(10))

print("\nFTP protocols:")
print(Counter(
    e.get("connection", {}).get("protocol")
    for e in ftp_events
).most_common(10))

Total precise FTP events: 76

FTP types:
Counter({'Dionaea': 76})

FTP destination ports:
[(21, 76)]

FTP protocols:
[('ftpd', 76)]


In [ ]:
print("Number of FTP events:", len(ftp_events))

all_keys = set()

for event in ftp_events:
    all_keys.update(event.keys())

print("\nNumber of unique fields:", len(all_keys))

print("\nFields:")
for key in sorted(all_keys):
    print("-", key)

Number of FTP events: 76

Number of unique fields: 22

Fields:
- @timestamp
- @version
- connection
- dest_ip
- dest_port
- ftp
- geoip
- geoip_ext
- headers
- host
- ip_rep
- password
- path
- src_hostname
- src_ip
- src_port
- t-pot_hostname
- t-pot_ip_ext
- t-pot_ip_int
- timestamp
- type
- username


In [ ]:
print("\nFTP fields:")
for event in ftp_events:
    if "ftp" in event:
        print(event["ftp"])


FTP fields:
{'commands': {'arguments': ['TLS'], 'command': ['AUTH']}}
{'commands': {'arguments': ['TLS', '', ''], 'command': ['AUTH', '\x16\x03\x01\x03S\x01', '']}}
{'commands': {'arguments': ['anonymous'], 'command': ['USER']}}
{'commands': {'arguments': ['TLS', 'SSL'], 'command': ['AUTH', 'AUTH']}}
{'commands': {'arguments': ['TLS', 'SSL'], 'command': ['AUTH', 'AUTH']}}
{'commands': {'arguments': ['TLS', 'SSL'], 'command': ['AUTH', 'AUTH']}}
{'commands': {'arguments': ['TLS', '', ''], 'command': ['AUTH', '\x16\x03\x01\x03S\x01', '']}}
{'commands': {'arguments': ['anonymous'], 'command': ['USER']}}
{'commands': {'arguments': ['TLS', 'SSL'], 'command': ['AUTH', 'AUTH']}}
{'commands': {'arguments': [''], 'command': ['MGLNDD_68.221.134.12_21']}}
{'commands': {'arguments': [''], 'command': ['MGLNDD_4.247.175.193_21']}}
{'commands': {'arguments': ['anonymous'], 'command': ['USER']}}
{'commands': {'arguments': ['TLS', 'SSL'], 'command': ['AUTH', 'AUTH']}}
{'commands': {'arguments': ['anony

In [ ]:
import pandas as pd

ftp_df = pd.json_normalize(ftp_events)

print("Shape:", ftp_df.shape)

missing = ftp_df.isna().sum().sort_values(ascending=False)

print("\nMissing values:")
print(missing)

print("\nMissing percentage:")
print((ftp_df.isna().mean() * 100).round(2).sort_values(ascending=False))

Shape: (76, 66)

Missing values:
headers.http_accept    76
username               72
password               72
geoip_ext.dma_code     65
geoip.dma_code         55
                       ..
geoip.ip                0
geoip.location.lon      0
geoip.location.lat      0
geoip.longitude         0
connection.protocol     0
Length: 66, dtype: int64

Missing percentage:
headers.http_accept    100.00
username                94.74
password                94.74
geoip_ext.dma_code      85.53
geoip.dma_code          72.37
                        ...  
geoip.ip                 0.00
geoip.location.lon       0.00
geoip.location.lat       0.00
geoip.longitude          0.00
connection.protocol      0.00
Length: 66, dtype: float64


In [ ]:
import json

# Convert each row to a comparable representation
row_strings = ftp_df.apply(
    lambda row: json.dumps(row.to_dict(), sort_keys=True, default=str),
    axis=1
)

duplicates = row_strings.duplicated().sum()

print("Total rows:", len(ftp_df))
print("Exact duplicate rows:", duplicates)

print(
    "Duplicate percentage:",
    round((duplicates / len(ftp_df)) * 100, 2),
    "%"
)

Total rows: 76
Exact duplicate rows: 0
Duplicate percentage: 0.0 %


In [ ]:
print("=== DATA TYPES ===")
print(ftp_df.dtypes)

print("\n=== TIMESTAMP CHECK ===")

for col in ["@timestamp", "timestamp"]:
    if col in ftp_df.columns:
        converted = pd.to_datetime(ftp_df[col], errors="coerce", utc=True)

        print(f"\n{col}")
        print("Valid:", converted.notna().sum())
        print("Invalid:", converted.isna().sum())
        print("Min:", converted.min())
        print("Max:", converted.max())

=== DATA TYPES ===
t-pot_hostname      object
t-pot_ip_int        object
t-pot_ip_ext        object
src_port             int64
@version            object
                    ...   
geoip.dma_code     float64
geoip.city_name     object
ip_rep              object
password            object
username            object
Length: 66, dtype: object

=== TIMESTAMP CHECK ===

@timestamp
Valid: 76
Invalid: 0
Min: 2025-06-09 20:06:14.496000+00:00
Max: 2025-06-11 15:33:07.648000+00:00

timestamp
Valid: 76
Invalid: 0
Min: 2025-06-09 20:06:14.496363+00:00
Max: 2025-06-11 15:33:07.648357+00:00


In [ ]:
missing = ftp_df.isna().sum()
missing_pct = (missing / len(ftp_df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct
})

print(missing_table[missing_table["missing_count"] > 0]
      .sort_values("missing_percentage", ascending=False))

                        missing_count  missing_percentage
headers.http_accept                76              100.00
password                           72               94.74
username                           72               94.74
geoip_ext.dma_code                 65               85.53
geoip.dma_code                     55               72.37
geoip.postal_code                  40               52.63
ip_rep                             38               50.00
geoip.city_name                    35               46.05
geoip.region_code                  32               42.11
geoip.region_name                  32               42.11
ftp.commands.command               23               30.26
ftp.commands.arguments             23               30.26


In [ ]:
missing = ftp_df.isna().sum()
missing_pct = (missing / len(ftp_df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct
})

print(missing_table.sort_values("missing_percentage", ascending=False).to_string())

                          missing_count  missing_percentage
headers.http_accept                  76              100.00
username                             72               94.74
password                             72               94.74
geoip_ext.dma_code                   65               85.53
geoip.dma_code                       55               72.37
geoip.postal_code                    40               52.63
ip_rep                               38               50.00
geoip.city_name                      35               46.05
geoip.region_code                    32               42.11
geoip.region_name                    32               42.11
ftp.commands.arguments               23               30.26
ftp.commands.command                 23               30.26
t-pot_ip_int                          0                0.00
t-pot_hostname                        0                0.00
t-pot_ip_ext                          0                0.00
src_port                              0 

In [ ]:
print("=== USERNAME ===")
print(ftp_df["username"].dropna().value_counts().head(20))

print("\n=== PASSWORD ===")
print(ftp_df["password"].dropna().value_counts().head(20))

print("\n=== FTP COMMANDS ===")
print(ftp_df["ftp.commands.command"].dropna().to_string(index=False))

print("\n=== FTP ARGUMENTS ===")
print(ftp_df["ftp.commands.arguments"].dropna().to_string(index=False))

print("\n=== IP REP ===")
print(ftp_df["ip_rep"].dropna().value_counts())

=== USERNAME ===
username
anonymous    4
Name: count, dtype: int64

=== PASSWORD ===
password
abuse@onyphe.io    2
anonymous          1
anonymous@         1
Name: count, dtype: int64

=== FTP COMMANDS ===
                                    [AUTH]
                          [AUTH, S, ]
                                    [USER]
                              [AUTH, AUTH]
                              [AUTH, AUTH]
                              [AUTH, AUTH]
                          [AUTH, S, ]
                                    [USER]
                              [AUTH, AUTH]
                 [MGLNDD_68.221.134.12_21]
                 [MGLNDD_4.247.175.193_21]
                                    [USER]
                              [AUTH, AUTH]
                              [USER, PASS]
               [GET, Host:, HELP, EHLO, ?]
                              [AUTH, AUTH]
                        [AUTH, USER, PASS]
                                    [USER]
[USER, PASS, FEAT, TY

In [ ]:
print("Unique source IPs:",
      ftp_df["src_ip"].nunique())

print("\nTop source IPs:")
print(ftp_df["src_ip"].value_counts().head(20))

Unique source IPs: 53

Top source IPs:
src_ip
35.216.227.70      4
156.229.16.165     4
118.26.111.94      4
152.32.245.186     3
71.6.232.20        3
45.82.78.254       2
4.154.233.151      2
20.175.98.249      2
172.236.228.224    2
205.210.31.12      2
20.65.193.113      2
106.75.35.191      2
135.119.184.174    2
45.79.181.179      2
106.75.50.248      2
20.38.33.1         1
198.235.24.69      1
121.225.148.195    1
135.237.127.223    1
64.62.197.90       1
Name: count, dtype: int64


In [ ]:
from collections import Counter

commands = []

for value in ftp_df["ftp.commands.command"].dropna():
    if isinstance(value, list):
        commands.extend(value)

print(Counter(commands).most_common(20))

[('AUTH', 47), ('USER', 11), ('HELP', 10), ('PASS', 7), ('FEAT', 5), ('QUIT', 4), ('\x16\x03\x01\x01', 4), ('SYST', 3), ('\x16\x03\x01\x03S\x01', 2), ('', 2), ('\x01', 2), ('MGLNDD_68.221.134.12_21', 1), ('MGLNDD_4.247.175.193_21', 1), ('GET', 1), ('Host:', 1), ('EHLO', 1), ('?', 1), ('TYPE', 1), ('EPSV', 1), ('PASV', 1)]


In [ ]:
print(ftp_df.columns.tolist())

['t-pot_hostname', 't-pot_ip_int', 't-pot_ip_ext', 'src_port', '@version', 'dest_port', 'timestamp', '@timestamp', 'path', 'host', 'dest_ip', 'src_hostname', 'type', 'src_ip', 'geoip_ext.timezone', 'geoip_ext.latitude', 'geoip_ext.postal_code', 'geoip_ext.country_code2', 'geoip_ext.country_name', 'geoip_ext.country_code3', 'geoip_ext.region_code', 'geoip_ext.as_org', 'geoip_ext.continent_code', 'geoip_ext.region_name', 'geoip_ext.city_name', 'geoip_ext.longitude', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.ip', 'geoip_ext.asn', 'headers.request_method', 'headers.content_type', 'headers.request_path', 'headers.http_version', 'headers.http_user_agent', 'headers.content_length', 'headers.connection', 'headers.http_host', 'headers.accept_encoding', 'headers.http_accept', 'ftp.commands.arguments', 'ftp.commands.command', 'geoip.timezone', 'geoip.latitude', 'geoip.country_code2', 'geoip.country_name', 'geoip.country_code3', 'geoip.region_code', 'geoip.as_org', 'geoip.cont

In [ ]:
print("Total events:", len(ftp_df))

connection_cols = [
    "src_ip",
    "src_port",
    "dest_ip",
    "dest_port"
]

print("\nUnique connections:")
print(ftp_df[connection_cols].drop_duplicates().shape[0])

print("\nRepeated connections:")
conn_counts = (
    ftp_df.groupby(connection_cols)
    .size()
    .sort_values(ascending=False)
)

print(conn_counts.head(20))

Total events: 76

Unique connections:
75

Repeated connections:
src_ip           src_port  dest_ip     dest_port
205.210.31.12    62076     172.19.0.2  21           2
106.75.35.191    56320     172.20.0.2  21           1
103.203.59.5     46536     172.19.0.2  21           1
106.75.50.248    36978     172.20.0.2  21           1
                 37444     172.20.0.2  21           1
118.26.111.94    37818     172.21.0.2  21           1
                 37820     172.21.0.2  21           1
                 55048     172.21.0.2  21           1
                 55060     172.21.0.2  21           1
121.225.148.195  14731     172.20.0.2  21           1
106.75.35.191    56434     172.20.0.2  21           1
135.119.184.174  59658     172.19.0.2  21           1
                 59716     172.19.0.2  21           1
147.185.132.42   57540     172.21.0.2  21           1
135.237.127.223  54714     172.19.0.2  21           1
152.32.245.186   59916     172.20.0.2  21           1
                 60270 

In [ ]:
print("=== FTP AUTHENTICATION ANALYSIS ===")

# Username attempts
print("\nUnique usernames:")
print(ftp_df["username"].dropna().value_counts())

# Password attempts
print("\nUnique passwords:")
print(ftp_df["password"].dropna().value_counts())

# Events containing USER command
user_events = ftp_df[
    ftp_df["ftp.commands.command"].apply(
        lambda x: isinstance(x, list) and "USER" in x
    )
]

print("\nEvents containing USER command:", len(user_events))

# Events containing PASS command
pass_events = ftp_df[
    ftp_df["ftp.commands.command"].apply(
        lambda x: isinstance(x, list) and "PASS" in x
    )
]

print("Events containing PASS command:", len(pass_events))

# Events containing AUTH command
auth_events = ftp_df[
    ftp_df["ftp.commands.command"].apply(
        lambda x: isinstance(x, list) and "AUTH" in x
    )
]

print("Events containing AUTH command:", len(auth_events))

=== FTP AUTHENTICATION ANALYSIS ===

Unique usernames:
username
anonymous    4
Name: count, dtype: int64

Unique passwords:
password
abuse@onyphe.io    2
anonymous          1
anonymous@         1
Name: count, dtype: int64

Events containing USER command: 11
Events containing PASS command: 7
Events containing AUTH command: 29


In [ ]:
print("=== IP / GEOIP ANALYSIS ===")

print("\nUnique source IPs:")
print(ftp_df["src_ip"].nunique())

print("\nTop source countries:")
print(ftp_df["geoip.country_name"].value_counts().head(15))

print("\nTop source country codes:")
print(ftp_df["geoip.country_code2"].value_counts().head(15))

print("\nTop AS Organizations:")
print(ftp_df["geoip.as_org"].value_counts().head(15))

print("\nIP Reputation:")
print(ftp_df["ip_rep"].value_counts(dropna=False))

print("\nTop source IPs:")
print(ftp_df["src_ip"].value_counts().head(15))

=== IP / GEOIP ANALYSIS ===

Unique source IPs:
53

Top source countries:
geoip.country_name
United States     46
China              7
France             5
Switzerland        4
Singapore          4
Thailand           3
Germany            2
United Kingdom     2
Canada             2
Bulgaria           1
Name: count, dtype: int64

Top source country codes:
geoip.country_code2
US    46
CN     7
FR     5
CH     4
SG     4
TH     3
DE     2
GB     2
CA     2
BG     1
Name: count, dtype: int64

Top AS Organizations:
geoip.as_org
MICROSOFT-CORP-MSN-AS-BLOCK                 17
UCLOUD INFORMATION TECHNOLOGY HK LIMITED    11
GOOGLE-CLOUD-PLATFORM                       10
China Unicom Beijing Province Network        4
Akamai Connected Cloud                       4
HURRICANE                                    4
ONYPHE SAS                                   4
GOOGLE                                       4
CARINET                                      3
CENSYS-ARIN-01                               3
Dr

In [ ]:
print("=== FTP COMMAND EDA ===")

from collections import Counter

command_counter = Counter()

for commands in ftp_df["ftp.commands.command"].dropna():
    if isinstance(commands, list):
        for cmd in commands:
            if cmd is not None:
                command_counter[str(cmd)] += 1

print("\nAll FTP commands:")
print(command_counter)

print("\nTop 10 commands:")
print(command_counter.most_common(10))

=== FTP COMMAND EDA ===

All FTP commands:
Counter({'AUTH': 47, 'USER': 11, 'HELP': 10, 'PASS': 7, 'FEAT': 5, 'QUIT': 4, '\x16\x03\x01\x01': 4, 'SYST': 3, '\x16\x03\x01\x03S\x01': 2, '': 2, '\x01': 2, 'MGLNDD_68.221.134.12_21': 1, 'MGLNDD_4.247.175.193_21': 1, 'GET': 1, 'Host:': 1, 'EHLO': 1, '?': 1, 'TYPE': 1, 'EPSV': 1, 'PASV': 1, 'STAT': 1, '\x16\x03\x01\x01\x17\x01': 1, 'MGLNDD_20.87.74.66_21': 1, 'MGLNDD_68.221.197.138_21': 1})

Top 10 commands:
[('AUTH', 47), ('USER', 11), ('HELP', 10), ('PASS', 7), ('FEAT', 5), ('QUIT', 4), ('\x16\x03\x01\x01', 4), ('SYST', 3), ('\x16\x03\x01\x03S\x01', 2), ('', 2)]


In [ ]:
print("=== FTP COMMAND BEHAVIOR ===")

ftp_df["command_count"] = ftp_df["ftp.commands.command"].apply(
    lambda x: len(x) if isinstance(x, list) else 0
)

print("\nCommand count per event:")
print(ftp_df["command_count"].describe())

print("\nDistribution of command counts:")
print(ftp_df["command_count"].value_counts().sort_index())

print("\nEvents with commands:")
print((ftp_df["command_count"] > 0).sum())

print("\nEvents without commands:")
print((ftp_df["command_count"] == 0).sum())

=== FTP COMMAND BEHAVIOR ===

Command count per event:
count    76.000000
mean      1.447368
std       1.398997
min       0.000000
25%       0.000000
50%       1.000000
75%       2.000000
max       7.000000
Name: command_count, dtype: float64

Distribution of command counts:
command_count
0    23
1    19
2    22
3     5
4     5
5     1
7     1
Name: count, dtype: int64

Events with commands:
53

Events without commands:
23


In [ ]:
print("=== FTP CONNECTION EDA ===")

print("\nConnection types:")
print(ftp_df["connection.type"].value_counts(dropna=False))

print("\nConnection transports:")
print(ftp_df["connection.transport"].value_counts(dropna=False))

print("\nConnection protocols:")
print(ftp_df["connection.protocol"].value_counts(dropna=False))

print("\nDestination IPs:")
print(ftp_df["dest_ip"].value_counts())

print("\nDestination ports:")
print(ftp_df["dest_port"].value_counts())

=== FTP CONNECTION EDA ===

Connection types:
connection.type
accept    76
Name: count, dtype: int64

Connection transports:
connection.transport
tcp    76
Name: count, dtype: int64

Connection protocols:
connection.protocol
ftpd    76
Name: count, dtype: int64

Destination IPs:
dest_ip
172.19.0.2    30
172.21.0.2    18
172.20.0.2    18
172.18.0.2    10
Name: count, dtype: int64

Destination ports:
dest_port
21    76
Name: count, dtype: int64


In [ ]:
print("=== FTP SOURCE BEHAVIOR ===")

print("\nUnique source IPs:")
print(ftp_df["src_ip"].nunique())

print("\nTop source IPs:")
print(ftp_df["src_ip"].value_counts().head(20))

print("\nSource port statistics:")
print(ftp_df["src_port"].describe())

print("\nUnique source ports:")
print(ftp_df["src_port"].nunique())

print("\nRepeated source IPs:")
src_counts = ftp_df["src_ip"].value_counts()
print(src_counts[src_counts > 1])

=== FTP SOURCE BEHAVIOR ===

Unique source IPs:
53

Top source IPs:
src_ip
35.216.227.70      4
156.229.16.165     4
118.26.111.94      4
152.32.245.186     3
71.6.232.20        3
45.82.78.254       2
4.154.233.151      2
20.175.98.249      2
172.236.228.224    2
205.210.31.12      2
20.65.193.113      2
106.75.35.191      2
135.119.184.174    2
45.79.181.179      2
106.75.50.248      2
20.38.33.1         1
198.235.24.69      1
121.225.148.195    1
135.237.127.223    1
64.62.197.90       1
Name: count, dtype: int64

Source port statistics:
count       76.000000
mean     46297.815789
std      13057.233138
min       7384.000000
25%      37761.500000
50%      49045.000000
75%      56308.750000
max      64594.000000
Name: src_port, dtype: float64

Unique source ports:
75

Repeated source IPs:
src_ip
35.216.227.70      4
156.229.16.165     4
118.26.111.94      4
152.32.245.186     3
71.6.232.20        3
45.82.78.254       2
4.154.233.151      2
20.175.98.249      2
172.236.228.224    2
205.

In [ ]:
print("=== FTP AUTHENTICATION BEHAVIOR ===")

def get_commands(x):
    if isinstance(x, list):
        return [str(c).strip() for c in x if c is not None]
    return []

ftp_df["commands_clean"] = ftp_df["ftp.commands.command"].apply(get_commands)

ftp_df["has_USER"] = ftp_df["commands_clean"].apply(lambda x: "USER" in x)
ftp_df["has_PASS"] = ftp_df["commands_clean"].apply(lambda x: "PASS" in x)
ftp_df["has_AUTH"] = ftp_df["commands_clean"].apply(lambda x: "AUTH" in x)

print("\nUSER events:", ftp_df["has_USER"].sum())
print("PASS events:", ftp_df["has_PASS"].sum())
print("AUTH events:", ftp_df["has_AUTH"].sum())

print("\nAuthentication combinations:")
print(
    ftp_df[["has_AUTH", "has_USER", "has_PASS"]]
    .value_counts()
    .sort_index()
)

print("\nEvents containing USER + PASS:")
print((ftp_df["has_USER"] & ftp_df["has_PASS"]).sum())

print("\nEvents containing USER + PASS + AUTH:")
print(
    (ftp_df["has_USER"] &
     ftp_df["has_PASS"] &
     ftp_df["has_AUTH"]).sum()
)

=== FTP AUTHENTICATION BEHAVIOR ===

USER events: 11
PASS events: 7
AUTH events: 29

Authentication combinations:
has_AUTH  has_USER  has_PASS
False     False     False       38
          True      False        4
                    True         5
True      False     False       27
          True      True         2
Name: count, dtype: int64

Events containing USER + PASS:
7

Events containing USER + PASS + AUTH:
2


In [ ]:
print("=== FTP COLUMN CLASSIFICATION ===")

for col in ftp_df.columns:
    missing = ftp_df[col].isna().sum()

    try:
        unique = ftp_df[col].nunique(dropna=True)
    except TypeError:
        unique = "complex"

    print(f"{col:40} | missing={missing:2} | unique={str(unique):>7}")

=== FTP COLUMN CLASSIFICATION ===
t-pot_hostname                           | missing= 0 | unique=      4
t-pot_ip_int                             | missing= 0 | unique=      4
t-pot_ip_ext                             | missing= 0 | unique=     10
src_port                                 | missing= 0 | unique=     75
@version                                 | missing= 0 | unique=      1
dest_port                                | missing= 0 | unique=      1
timestamp                                | missing= 0 | unique=     75
@timestamp                               | missing= 0 | unique=     75
path                                     | missing= 0 | unique=      1
host                                     | missing= 0 | unique=      2
dest_ip                                  | missing= 0 | unique=      4
src_hostname                             | missing= 0 | unique=      1
type                                     | missing= 0 | unique=      1
src_ip                                   | 

In [ ]:
# Proposed columns for removal - DO NOT DROP YET

drop_candidates = [
    '@version',
    'path',
    'src_hostname',
    'type',
    'connection.type',
    'connection.transport',
    'connection.protocol',
    'headers.http_accept',
]

print("DROP CANDIDATES:")
for col in drop_candidates:
    print(f"- {col}")

print("\nNumber of candidates:", len(drop_candidates))

print("\nOriginal shape:", ftp_df.shape)
print("Shape if dropped:",
      (ftp_df.shape[0], ftp_df.shape[1] - len(drop_candidates)))

DROP CANDIDATES:
- @version
- path
- src_hostname
- type
- connection.type
- connection.transport
- connection.protocol
- headers.http_accept

Number of candidates: 8

Original shape: (76, 71)
Shape if dropped: (76, 63)


In [ ]:
drop_candidates = [
    '@version',
    'path',
    'src_hostname',
    'type',
    'connection.type',
    'connection.transport',
    'connection.protocol',
    'headers.http_accept'
]

ftp_df_clean = ftp_df.drop(columns=drop_candidates)

print("Original shape:", ftp_df.shape)
print("Cleaned shape:", ftp_df_clean.shape)

print("\nRemaining columns:")
print(ftp_df_clean.columns.tolist())

Original shape: (76, 71)
Cleaned shape: (76, 63)

Remaining columns:
['t-pot_hostname', 't-pot_ip_int', 't-pot_ip_ext', 'src_port', 'dest_port', 'timestamp', '@timestamp', 'host', 'dest_ip', 'src_ip', 'geoip_ext.timezone', 'geoip_ext.latitude', 'geoip_ext.postal_code', 'geoip_ext.country_code2', 'geoip_ext.country_name', 'geoip_ext.country_code3', 'geoip_ext.region_code', 'geoip_ext.as_org', 'geoip_ext.continent_code', 'geoip_ext.region_name', 'geoip_ext.city_name', 'geoip_ext.longitude', 'geoip_ext.location.lat', 'geoip_ext.location.lon', 'geoip_ext.ip', 'geoip_ext.asn', 'headers.request_method', 'headers.content_type', 'headers.request_path', 'headers.http_version', 'headers.http_user_agent', 'headers.content_length', 'headers.connection', 'headers.http_host', 'headers.accept_encoding', 'ftp.commands.arguments', 'ftp.commands.command', 'geoip.timezone', 'geoip.latitude', 'geoip.country_code2', 'geoip.country_name', 'geoip.country_code3', 'geoip.region_code', 'geoip.as_org', 'geoip.co

In [ ]:
ftp_clean = ftp_df.drop(columns=[
    '@version',
    'path',
    'src_hostname',
    'type',
    'connection.type',
    'connection.transport',
    'connection.protocol',
    'headers.http_accept'
]).copy()

print("Original shape:", ftp_df.shape)
print("Cleaned shape:", ftp_clean.shape)

Original shape: (76, 71)
Cleaned shape: (76, 63)


In [ ]:
constant_cols = []

for col in ftp_clean.columns:
    try:
        if ftp_clean[col].nunique(dropna=False) <= 1:
            constant_cols.append(col)
    except TypeError:
        pass

print("CONSTANT COLUMNS:")

for col in constant_cols:
    print("-", col)

print("\nNumber of constant columns:", len(constant_cols))

CONSTANT COLUMNS:
- dest_port
- geoip_ext.as_org
- geoip_ext.asn
- headers.request_method
- headers.content_type
- headers.request_path
- headers.http_version
- headers.http_user_agent
- headers.connection
- headers.http_host
- headers.accept_encoding

Number of constant columns: 11


In [ ]:
print("=== FTP COLUMNS BEFORE CONSTANT REMOVAL ===")

constant_cols = [
    'dest_port',
    'geoip_ext.as_org',
    'geoip_ext.asn',
    'headers.request_method',
    'headers.content_type',
    'headers.request_path',
    'headers.http_version',
    'headers.http_user_agent',
    'headers.connection',
    'headers.http_host',
    'headers.accept_encoding'
]

for col in constant_cols:
    if col in ftp_clean.columns:
        print(f"{col}:")
        print("  unique:", ftp_clean[col].nunique(dropna=False))
        print("  value:", ftp_clean[col].drop_duplicates().tolist()[:5])

=== FTP COLUMNS BEFORE CONSTANT REMOVAL ===
dest_port:
  unique: 1
  value: [21]
geoip_ext.as_org:
  unique: 1
  value: ['MICROSOFT-CORP-MSN-AS-BLOCK']
geoip_ext.asn:
  unique: 1
  value: [8075]
headers.request_method:
  unique: 1
  value: ['POST']
headers.content_type:
  unique: 1
  value: ['application/json']
headers.request_path:
  unique: 1
  value: ['/']
headers.http_version:
  unique: 1
  value: ['HTTP/1.0']
headers.http_user_agent:
  unique: 1
  value: ['Manticore 0.9.1']
headers.connection:
  unique: 1
  value: ['close']
headers.http_host:
  unique: 1
  value: ['logstash:64305']
headers.accept_encoding:
  unique: 1
  value: ['gzip,deflate']


In [ ]:
constant_cols = [
    'dest_port',
    'geoip_ext.as_org',
    'geoip_ext.asn',
    'headers.request_method',
    'headers.content_type',
    'headers.request_path',
    'headers.http_version',
    'headers.http_user_agent',
    'headers.connection',
    'headers.http_host',
    'headers.accept_encoding'
]

ftp_clean = ftp_clean.drop(columns=constant_cols)

print("Shape after constant-column removal:", ftp_clean.shape)
print("\nRemaining columns:", len(ftp_clean.columns))

Shape after constant-column removal: (76, 52)

Remaining columns: 52


In [ ]:
print("=== MISSING VALUES AFTER CLEANING ===")

missing_after = (
    ftp_clean.isna()
    .sum()
    .to_frame("missing_count")
)

missing_after["missing_percentage"] = (
    missing_after["missing_count"] / len(ftp_clean) * 100
)

missing_after = missing_after.sort_values(
    "missing_percentage",
    ascending=False
)

print(missing_after.to_string())

=== MISSING VALUES AFTER CLEANING ===
                          missing_count  missing_percentage
username                             72           94.736842
password                             72           94.736842
geoip_ext.dma_code                   65           85.526316
geoip.dma_code                       55           72.368421
geoip.postal_code                    40           52.631579
ip_rep                               38           50.000000
geoip.city_name                      35           46.052632
geoip.region_name                    32           42.105263
geoip.region_code                    32           42.105263
ftp.commands.command                 23           30.263158
ftp.commands.arguments               23           30.263158
src_ip                                0            0.000000
timestamp                             0            0.000000
@timestamp                            0            0.000000
host                                  0            0.000000
de

In [ ]:
print("=== GEOIP EXT SUMMARY ===")

for col in [
    "geoip_ext.ip",
    "geoip_ext.country_name",
    "geoip_ext.country_code2",
    "geoip_ext.region_name",
    "geoip_ext.city_name",
    "geoip_ext.timezone",
    "geoip_ext.latitude",
    "geoip_ext.longitude"
]:
    if col in ftp_clean.columns:
        print(
            f"{col:35} | "
            f"missing={ftp_clean[col].isna().sum():2} | "
            f"unique={ftp_clean[col].nunique(dropna=True):2}"
        )

=== GEOIP EXT SUMMARY ===
geoip_ext.ip                        | missing= 0 | unique=10
geoip_ext.country_name              | missing= 0 | unique= 4
geoip_ext.country_code2             | missing= 0 | unique= 4
geoip_ext.region_name               | missing= 0 | unique= 4
geoip_ext.city_name                 | missing= 0 | unique= 4
geoip_ext.timezone                  | missing= 0 | unique= 4
geoip_ext.latitude                  | missing= 0 | unique= 4
geoip_ext.longitude                 | missing= 0 | unique= 4


In [ ]:
print("=== TIMESTAMP DIFFERENCE ===")

ts1 = pd.to_datetime(ftp_clean["timestamp"], errors="coerce", utc=True)
ts2 = pd.to_datetime(ftp_clean["@timestamp"], errors="coerce", utc=True)

time_diff = (ts2 - ts1).dt.total_seconds()

print("\nDifference statistics:")
print(time_diff.describe())

print("\nUnique differences:")
print(time_diff.value_counts().head(20))

=== TIMESTAMP DIFFERENCE ===

Difference statistics:
count    76.000000
mean     -0.000501
std       0.000287
min      -0.000994
25%      -0.000755
50%      -0.000454
75%      -0.000255
max      -0.000009
dtype: float64

Unique differences:
-0.000378    3
-0.000357    1
-0.000313    1
-0.000402    1
-0.000930    1
-0.000292    1
-0.000490    1
-0.000778    1
-0.000874    1
-0.000591    1
-0.000344    1
-0.000009    1
-0.000205    1
-0.000775    1
-0.000220    1
-0.000196    1
-0.000611    1
-0.000114    1
-0.000223    1
-0.000256    1
Name: count, dtype: int64


In [ ]:
ftp_clean = ftp_clean.drop(columns=["timestamp"])

print("Shape:", ftp_clean.shape)
print("\nTimestamp columns remaining:")

print([
    col for col in ftp_clean.columns
    if "timestamp" in col.lower()
])

Shape: (76, 51)

Timestamp columns remaining:
['@timestamp']


In [ ]:
print("=== CLEAN FTP COMMANDS SAMPLE ===")

for i, row in ftp_clean["commands_clean"].head(20).items():
    print(f"Event {i}: {row}")

=== CLEAN FTP COMMANDS SAMPLE ===
Event 0: ['AUTH']
Event 1: ['AUTH', '\x16\x03\x01\x03S\x01', '']
Event 2: ['USER']
Event 3: ['AUTH', 'AUTH']
Event 4: ['AUTH', 'AUTH']
Event 5: ['AUTH', 'AUTH']
Event 6: ['AUTH', '\x16\x03\x01\x03S\x01', '']
Event 7: ['USER']
Event 8: ['AUTH', 'AUTH']
Event 9: ['MGLNDD_68.221.134.12_21']
Event 10: ['MGLNDD_4.247.175.193_21']
Event 11: ['USER']
Event 12: []
Event 13: ['AUTH', 'AUTH']
Event 14: ['USER', 'PASS']
Event 15: []
Event 16: ['GET', 'Host:', 'HELP', 'EHLO', '?']
Event 17: []
Event 18: ['AUTH', 'AUTH']
Event 19: ['AUTH', 'USER', 'PASS']


In [ ]:
from collections import Counter

print("=== RAW COMMAND TOKENS ===")

all_tokens = []

for commands in ftp_clean["commands_clean"]:
    if isinstance(commands, list):
        all_tokens.extend(commands)

command_counter = Counter(all_tokens)

for cmd, count in command_counter.most_common():
    print(repr(cmd), ":", count)

=== RAW COMMAND TOKENS ===
'AUTH' : 47
'USER' : 11
'HELP' : 10
'PASS' : 7
'FEAT' : 5
'QUIT' : 4
'\x16\x03\x01\x01' : 4
'SYST' : 3
'\x16\x03\x01\x03S\x01' : 2
'' : 2
'\x01' : 2
'MGLNDD_68.221.134.12_21' : 1
'MGLNDD_4.247.175.193_21' : 1
'GET' : 1
'Host:' : 1
'EHLO' : 1
'?' : 1
'TYPE' : 1
'EPSV' : 1
'PASV' : 1
'STAT' : 1
'\x16\x03\x01\x01\x17\x01' : 1
'MGLNDD_20.87.74.66_21' : 1
'MGLNDD_68.221.197.138_21' : 1


In [ ]:
VALID_FTP_COMMANDS = {
    "AUTH",
    "USER",
    "PASS",
    "HELP",
    "FEAT",
    "QUIT",
    "SYST",
    "TYPE",
    "EPSV",
    "PASV",
    "STAT"
}

ftp_clean["ftp_commands_clean"] = ftp_clean["commands_clean"].apply(
    lambda cmds: [
        cmd for cmd in cmds
        if cmd in VALID_FTP_COMMANDS
    ] if isinstance(cmds, list) else []
)

print("=== CLEAN FTP COMMANDS ===")

counter = Counter()

for cmds in ftp_clean["ftp_commands_clean"]:
    counter.update(cmds)

print(counter)

print("\nSample:")
for i, cmds in ftp_clean["ftp_commands_clean"].head(20).items():
    print(f"Event {i}: {cmds}")

=== CLEAN FTP COMMANDS ===
Counter({'AUTH': 47, 'USER': 11, 'HELP': 10, 'PASS': 7, 'FEAT': 5, 'QUIT': 4, 'SYST': 3, 'TYPE': 1, 'EPSV': 1, 'PASV': 1, 'STAT': 1})

Sample:
Event 0: ['AUTH']
Event 1: ['AUTH']
Event 2: ['USER']
Event 3: ['AUTH', 'AUTH']
Event 4: ['AUTH', 'AUTH']
Event 5: ['AUTH', 'AUTH']
Event 6: ['AUTH']
Event 7: ['USER']
Event 8: ['AUTH', 'AUTH']
Event 9: []
Event 10: []
Event 11: ['USER']
Event 12: []
Event 13: ['AUTH', 'AUTH']
Event 14: ['USER', 'PASS']
Event 15: []
Event 16: ['HELP']
Event 17: []
Event 18: ['AUTH', 'AUTH']
Event 19: ['AUTH', 'USER', 'PASS']


In [ ]:
ftp_clean["command_count"] = ftp_clean["ftp_commands_clean"].apply(len)

print("=== UPDATED COMMAND COUNT ===")

print(ftp_clean["command_count"].describe())

print("\nDistribution:")
print(ftp_clean["command_count"].value_counts().sort_index())

print("\nEvents with valid FTP commands:",
      (ftp_clean["command_count"] > 0).sum())

print("Events without valid FTP commands:",
      (ftp_clean["command_count"] == 0).sum())

=== UPDATED COMMAND COUNT ===
count    76.000000
mean      1.197368
std       1.366453
min       0.000000
25%       0.000000
50%       1.000000
75%       2.000000
max       7.000000
Name: command_count, dtype: float64

Distribution:
command_count
0    31
1    17
2    19
3     3
4     5
7     1
Name: count, dtype: int64

Events with valid FTP commands: 45
Events without valid FTP commands: 31


In [ ]:
# Recalculate authentication flags from cleaned FTP commands

ftp_clean["has_USER"] = ftp_clean["ftp_commands_clean"].apply(
    lambda cmds: int("USER" in cmds)
)

ftp_clean["has_PASS"] = ftp_clean["ftp_commands_clean"].apply(
    lambda cmds: int("PASS" in cmds)
)

ftp_clean["has_AUTH"] = ftp_clean["ftp_commands_clean"].apply(
    lambda cmds: int("AUTH" in cmds)
)

print("=== UPDATED FTP AUTHENTICATION FLAGS ===")

print("\nUSER events:", ftp_clean["has_USER"].sum())
print("PASS events:", ftp_clean["has_PASS"].sum())
print("AUTH events:", ftp_clean["has_AUTH"].sum())

print("\nAuthentication combinations:")
print(
    ftp_clean[
        ["has_AUTH", "has_USER", "has_PASS"]
    ].value_counts().sort_index()
)

print("\nUSER + PASS:",
      ((ftp_clean["has_USER"] == 1) &
       (ftp_clean["has_PASS"] == 1)).sum())

print("\nUSER + PASS + AUTH:",
      ((ftp_clean["has_USER"] == 1) &
       (ftp_clean["has_PASS"] == 1) &
       (ftp_clean["has_AUTH"] == 1)).sum())

=== UPDATED FTP AUTHENTICATION FLAGS ===

USER events: 11
PASS events: 7
AUTH events: 29

Authentication combinations:
has_AUTH  has_USER  has_PASS
0         0         0           38
          1         0            4
                    1            5
1         0         0           27
          1         1            2
Name: count, dtype: int64

USER + PASS: 7

USER + PASS + AUTH: 2


In [ ]:
print("=== FTP-SPECIFIC COLUMN AUDIT ===\n")

ftp_cols = [
    "ftp.commands.command",
    "ftp.commands.arguments",
    "commands_clean",
    "command_count",
    "has_USER",
    "has_PASS",
    "has_AUTH",
    "username",
    "password"
]

for col in ftp_cols:
    if col in ftp_clean.columns:
        print(f"\n--- {col} ---")
        print("dtype:", ftp_clean[col].dtype)
        print("missing:", ftp_clean[col].isna().sum())
        print("missing %:", round(ftp_clean[col].isna().mean() * 100, 2))

        try:
            print("unique:", ftp_clean[col].nunique(dropna=True))
        except:
            print("unique: complex/list data")

=== FTP-SPECIFIC COLUMN AUDIT ===


--- ftp.commands.command ---
dtype: object
missing: 23
missing %: 30.26
unique: complex/list data

--- ftp.commands.arguments ---
dtype: object
missing: 23
missing %: 30.26
unique: complex/list data

--- commands_clean ---
dtype: object
missing: 0
missing %: 0.0
unique: complex/list data

--- command_count ---
dtype: int64
missing: 0
missing %: 0.0
unique: 6

--- has_USER ---
dtype: int64
missing: 0
missing %: 0.0
unique: 2

--- has_PASS ---
dtype: int64
missing: 0
missing %: 0.0
unique: 2

--- has_AUTH ---
dtype: int64
missing: 0
missing %: 0.0
unique: 2

--- username ---
dtype: object
missing: 72
missing %: 94.74
unique: 1

--- password ---
dtype: object
missing: 72
missing %: 94.74
unique: 3


In [ ]:
print("=== USERNAME VALUES ===")
print(ftp_clean["username"].value_counts(dropna=False))

print("\n=== PASSWORD VALUES ===")
print(ftp_clean["password"].value_counts(dropna=False))

=== USERNAME VALUES ===
username
NaN          72
anonymous     4
Name: count, dtype: int64

=== PASSWORD VALUES ===
password
NaN                72
abuse@onyphe.io     2
anonymous           1
anonymous@          1
Name: count, dtype: int64


In [ ]:
# ==========================================
# FTP AUTHENTICATION FEATURES
# ==========================================

ftp_clean["has_username"] = ftp_clean["username"].notna().astype(int)
ftp_clean["has_password"] = ftp_clean["password"].notna().astype(int)

ftp_clean["is_anonymous"] = (
    ftp_clean["username"]
    .fillna("")
    .str.lower()
    .eq("anonymous")
    .astype(int)
)

print("=== NEW AUTHENTICATION FEATURES ===")

print("\nhas_username:")
print(ftp_clean["has_username"].value_counts())

print("\nhas_password:")
print(ftp_clean["has_password"].value_counts())

print("\nis_anonymous:")
print(ftp_clean["is_anonymous"].value_counts())

print("\n=== CHECK ===")
print(
    ftp_clean[
        ["username", "password",
         "has_username", "has_password", "is_anonymous"]
    ].dropna(how="all", subset=["username", "password"]).to_string(index=False)
)

=== NEW AUTHENTICATION FEATURES ===

has_username:
has_username
0    72
1     4
Name: count, dtype: int64

has_password:
has_password
0    72
1     4
Name: count, dtype: int64

is_anonymous:
is_anonymous
0    72
1     4
Name: count, dtype: int64

=== CHECK ===
 username        password  has_username  has_password  is_anonymous
anonymous abuse@onyphe.io             1             1             1
anonymous       anonymous             1             1             1
anonymous      anonymous@             1             1             1
anonymous abuse@onyphe.io             1             1             1


In [ ]:
# ==========================================
# DROP RAW FTP AUTH & COMMAND FIELDS
# ==========================================

drop_ftp_raw = [
    "ftp.commands.command",
    "ftp.commands.arguments",
    "commands_clean",
    "username",
    "password"
]

ftp_clean = ftp_clean.drop(
    columns=[c for c in drop_ftp_raw if c in ftp_clean.columns]
)

print("Dropped columns:")
print(drop_ftp_raw)

print("\nNew shape:", ftp_clean.shape)

print("\nRemaining FTP features:")
for col in [
    "command_count",
    "has_USER",
    "has_PASS",
    "has_AUTH",
    "has_username",
    "has_password",
    "is_anonymous"
]:
    if col in ftp_clean.columns:
        print("-", col)

Dropped columns:
['ftp.commands.command', 'ftp.commands.arguments', 'commands_clean', 'username', 'password']

New shape: (76, 50)

Remaining FTP features:
- command_count
- has_USER
- has_PASS
- has_AUTH
- has_username
- has_password
- is_anonymous


In [ ]:
print("=== FINAL MISSING VALUES REVIEW ===\n")

missing_review = pd.DataFrame({
    "missing_count": ftp_clean.isna().sum(),
    "missing_percentage": (
        ftp_clean.isna().mean() * 100
    ).round(2)
})

print(
    missing_review
    .sort_values("missing_percentage", ascending=False)
    .to_string()
)

=== FINAL MISSING VALUES REVIEW ===

                          missing_count  missing_percentage
geoip_ext.dma_code                   65               85.53
geoip.dma_code                       55               72.37
geoip.postal_code                    40               52.63
ip_rep                               38               50.00
geoip.city_name                      35               46.05
geoip.region_name                    32               42.11
geoip.region_code                    32               42.11
t-pot_ip_ext                          0                0.00
geoip_ext.timezone                    0                0.00
geoip_ext.latitude                    0                0.00
geoip_ext.postal_code                 0                0.00
src_port                              0                0.00
@timestamp                            0                0.00
host                                  0                0.00
dest_ip                               0                0.00
src

In [ ]:
print("=== GEOIP_EXT COLUMNS CURRENTLY AVAILABLE ===\n")

geoip_ext_cols = [
    col for col in ftp_clean.columns
    if col.startswith("geoip_ext.")
]

for col in geoip_ext_cols:
    print(
        f"{col:35} | "
        f"missing={ftp_clean[col].isna().sum():2} | "
        f"unique={ftp_clean[col].nunique(dropna=True):2}"
    )

=== GEOIP_EXT COLUMNS CURRENTLY AVAILABLE ===

geoip_ext.timezone                  | missing= 0 | unique= 4
geoip_ext.latitude                  | missing= 0 | unique= 4
geoip_ext.postal_code               | missing= 0 | unique= 4
geoip_ext.country_code2             | missing= 0 | unique= 4
geoip_ext.country_name              | missing= 0 | unique= 4
geoip_ext.country_code3             | missing= 0 | unique= 4
geoip_ext.region_code               | missing= 0 | unique= 4
geoip_ext.continent_code            | missing= 0 | unique= 4
geoip_ext.region_name               | missing= 0 | unique= 4
geoip_ext.city_name                 | missing= 0 | unique= 4
geoip_ext.longitude                 | missing= 0 | unique= 4
geoip_ext.location.lat              | missing= 0 | unique= 4
geoip_ext.location.lon              | missing= 0 | unique= 4
geoip_ext.ip                        | missing= 0 | unique=10
geoip_ext.dma_code                  | missing=65 | unique= 1


In [ ]:
print("=== SOURCE IP vs GEOIP_EXT IP ===\n")

comparison = ftp_clean[
    ["src_ip", "geoip.ip", "geoip_ext.ip", "geoip.country_name",
     "geoip_ext.country_name"]
].copy()

print(comparison.to_string(index=False))

=== SOURCE IP vs GEOIP_EXT IP ===

         src_ip        geoip.ip   geoip_ext.ip geoip.country_name geoip_ext.country_name
  198.235.24.69   198.235.24.69  4.221.107.209      United States           South Africa
  106.75.50.248   106.75.50.248  52.185.37.223              China          United States
  106.75.50.248   106.75.50.248  52.185.37.223              China          United States
     20.38.33.1      20.38.33.1  52.185.37.223      United States          United States
  20.65.193.113   20.65.193.113  4.221.107.209      United States           South Africa
  20.65.193.113   20.65.193.113  4.247.175.193      United States                  India
  106.75.35.191   106.75.35.191  68.221.134.12              China                  Spain
  106.75.35.191   106.75.35.191  68.221.134.12              China                  Spain
  20.65.193.213   20.65.193.213  68.221.134.12      United States                  Spain
 20.169.107.142  20.169.107.142  68.221.134.12      United States          

In [ ]:
print("=== GEOIP_EXT vs T-POT IPs ===\n")

cols = [
    "geoip_ext.ip",
    "t-pot_ip_ext",
    "t-pot_ip_int"
]

for col in cols:
    print(f"\n{col}")
    print("Unique:", ftp_clean[col].nunique())
    print(ftp_clean[col].value_counts().head(15))

=== GEOIP_EXT vs T-POT IPs ===


geoip_ext.ip
Unique: 10
geoip_ext.ip
68.221.197.138    24
20.87.74.66       18
135.235.171.89    14
68.221.134.12      5
52.173.49.247      3
52.185.37.223      3
40.86.6.196        3
4.221.107.209      2
4.247.175.193      2
52.176.6.22        2
Name: count, dtype: int64

t-pot_ip_ext
Unique: 10
t-pot_ip_ext
68.221.197.138    24
20.87.74.66       18
135.235.171.89    14
68.221.134.12      5
52.173.49.247      3
52.185.37.223      3
40.86.6.196        3
4.221.107.209      2
4.247.175.193      2
52.176.6.22        2
Name: count, dtype: int64

t-pot_ip_int
Unique: 4
t-pot_ip_int
10.3.0.4    29
10.4.0.4    20
10.1.0.4    16
10.2.0.4    11
Name: count, dtype: int64


In [ ]:
print("=== GEOIP REDUNDANCY CHECK ===\n")

pairs = [
    ("geoip.latitude", "geoip.location.lat"),
    ("geoip.longitude", "geoip.location.lon"),
    ("geoip.ip", "src_ip"),
]

for col1, col2 in pairs:
    same = (ftp_clean[col1] == ftp_clean[col2]).all()
    print(f"{col1}  vs  {col2}")
    print("Exactly identical:", same)
    print()

=== GEOIP REDUNDANCY CHECK ===

geoip.latitude  vs  geoip.location.lat
Exactly identical: True

geoip.longitude  vs  geoip.location.lon
Exactly identical: True

geoip.ip  vs  src_ip
Exactly identical: True



In [ ]:
print("=== GEOIP UNIQUE VALUES CHECK ===\n")

geoip_cols = [
    "geoip.country_code2",
    "geoip.country_name",
    "geoip.country_code3",
    "geoip.region_code",
    "geoip.region_name",
    "geoip.city_name",
    "geoip.as_org",
    "geoip.asn",
    "geoip.continent_code",
    "geoip.timezone",
    "geoip.latitude",
    "geoip.longitude",
    "geoip.postal_code",
    "geoip.dma_code"
]

for col in geoip_cols:
    if col in ftp_clean.columns:
        print(
            f"{col:30} | "
            f"missing={ftp_clean[col].isna().sum():2} | "
            f"unique={ftp_clean[col].nunique(dropna=True):3}"
        )

=== GEOIP UNIQUE VALUES CHECK ===

geoip.country_code2            | missing= 0 | unique= 10
geoip.country_name             | missing= 0 | unique= 10
geoip.country_code3            | missing= 0 | unique= 10
geoip.region_code              | missing=32 | unique= 14
geoip.region_name              | missing=32 | unique= 14
geoip.city_name                | missing=35 | unique= 16
geoip.as_org                   | missing= 0 | unique= 20
geoip.asn                      | missing= 0 | unique= 20
geoip.continent_code           | missing= 0 | unique=  3
geoip.timezone                 | missing= 0 | unique= 13
geoip.latitude                 | missing= 0 | unique= 22
geoip.longitude                | missing= 0 | unique= 22
geoip.postal_code              | missing=40 | unique= 14
geoip.dma_code                 | missing=55 | unique=  7


In [ ]:
print("=== GEOIP CATEGORICAL REDUNDANCY ===\n")

pairs = [
    ("geoip.country_name", "geoip.country_code2"),
    ("geoip.country_name", "geoip.country_code3"),
    ("geoip.region_code", "geoip.region_name"),
]

for col1, col2 in pairs:
    print(f"{col1} vs {col2}")

    mapping = (
        ftp_clean[[col1, col2]]
        .dropna()
        .drop_duplicates()
        .sort_values([col1, col2])
    )

    print("Unique pairs:", len(mapping))
    print(mapping.to_string(index=False))
    print()

=== GEOIP CATEGORICAL REDUNDANCY ===

geoip.country_name vs geoip.country_code2
Unique pairs: 10
geoip.country_name geoip.country_code2
          Bulgaria                  BG
            Canada                  CA
             China                  CN
            France                  FR
           Germany                  DE
         Singapore                  SG
       Switzerland                  CH
          Thailand                  TH
    United Kingdom                  GB
     United States                  US

geoip.country_name vs geoip.country_code3
Unique pairs: 10
geoip.country_name geoip.country_code3
          Bulgaria                  BG
            Canada                  CA
             China                  CN
            France                  FR
           Germany                  DE
         Singapore                  SG
       Switzerland                  CH
          Thailand                  TH
    United Kingdom                  GB
     United States      

In [ ]:
print("=== GEOIP ASN vs AS ORG ===\n")

mapping = (
    ftp_clean[["geoip.asn", "geoip.as_org"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["geoip.asn", "geoip.as_org"])
)

print("Unique ASN-Organization pairs:", len(mapping))
print(mapping.to_string(index=False))

print("\nUnique ASNs:", ftp_clean["geoip.asn"].nunique())
print("Unique Organizations:", ftp_clean["geoip.as_org"].nunique())

=== GEOIP ASN vs AS ORG ===

Unique ASN-Organization pairs: 20
 geoip.asn                             geoip.as_org
      4134                                 Chinanet
      4808    China Unicom Beijing Province Network
      4837           CHINA UNICOM China169 Backbone
      6939                                HURRICANE
      8075              MICROSOFT-CORP-MSN-AS-BLOCK
     10439                                  CARINET
     14061                         DIGITALOCEAN-ASN
     14618                               AMAZON-AES
     15169                                   GOOGLE
     16276                                  OVH SAS
     48090                      Techoff Srv Limited
     63949                   Akamai Connected Cloud
    135377 UCLOUD INFORMATION TECHNOLOGY HK LIMITED
    136180        Beijing Tiantexin Tech. Co., Ltd.
    211298                             Driftnet Ltd
    212512    Detai Prosperous Technologies Limited
    213412                               ONYPHE SAS
 

In [ ]:
print("=== GEOIP LOW-COVERAGE FEATURES ===\n")

cols = [
    "geoip.city_name",
    "geoip.postal_code",
    "geoip.dma_code",
    "geoip.continent_code",
    "geoip.timezone"
]

for col in cols:
    print(f"\n--- {col} ---")
    print("Missing:", ftp_clean[col].isna().sum())
    print("Non-missing:", ftp_clean[col].notna().sum())
    print("Unique:")
    print(ftp_clean[col].value_counts(dropna=False).head(20))

=== GEOIP LOW-COVERAGE FEATURES ===


--- geoip.city_name ---
Missing: 35
Non-missing: 41
Unique:
geoip.city_name
NaN             35
Phoenix          4
Zurich           4
Roubaix          4
Des Moines       4
Singapore        4
San Antonio      3
Bangkok          3
Washington       2
Cedar Knolls     2
Québec           2
London           2
Los Angeles      2
Saint Paul       2
Nanjing          1
Ashburn          1
North Bergen     1
Name: count, dtype: int64

--- geoip.postal_code ---
Missing: 40
Non-missing: 36
Unique:
geoip.postal_code
NaN      40
85036     4
8031      4
17        4
50307     4
78288     3
10200     3
07927     2
22747     2
WC2N      2
90060     2
55114     2
G1K       2
20149     1
07047     1
Name: count, dtype: int64

--- geoip.dma_code ---
Missing: 55
Non-missing: 21
Unique:
geoip.dma_code
NaN      55
753.0     4
679.0     4
641.0     3
511.0     3
501.0     3
803.0     2
613.0     2
Name: count, dtype: int64

--- geoip.continent_code ---
Missing: 0
Non-missing:

In [ ]:
print("=== GEOIP NUMERICAL / CATEGORICAL REDUNDANCY ===\n")

# Latitude / Longitude uniqueness
print("Latitude unique:", ftp_clean["geoip.latitude"].nunique())
print("Longitude unique:", ftp_clean["geoip.longitude"].nunique())

# Country -> Timezone relationship
country_timezone = (
    ftp_clean.groupby("geoip.country_code2")["geoip.timezone"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n=== TIMEZONE PER COUNTRY ===")
print(country_timezone)

# Country -> Region relationship
country_region = (
    ftp_clean.groupby("geoip.country_code2")["geoip.region_code"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n=== REGION PER COUNTRY ===")
print(country_region)

# Country -> City relationship
country_city = (
    ftp_clean.groupby("geoip.country_code2")["geoip.city_name"]
    .nunique()
    .sort_values(ascending=False)
)

print("\n=== CITY PER COUNTRY ===")
print(country_city)

# Coordinates consistency
coord_pairs = ftp_clean[
    ["geoip.latitude", "geoip.longitude"]
].drop_duplicates()

print("\n=== UNIQUE COORDINATE PAIRS ===")
print("Unique latitude-longitude pairs:", len(coord_pairs))
print(coord_pairs.sort_values(["geoip.latitude", "geoip.longitude"]).head(30))

=== GEOIP NUMERICAL / CATEGORICAL REDUNDANCY ===

Latitude unique: 22
Longitude unique: 22

=== TIMEZONE PER COUNTRY ===
geoip.country_code2
US    4
BG    1
CA    1
CH    1
DE    1
CN    1
FR    1
GB    1
SG    1
TH    1
Name: geoip.timezone, dtype: int64

=== REGION PER COUNTRY ===
geoip.country_code2
US    8
CA    1
TH    1
CH    1
FR    1
CN    1
GB    1
BG    0
DE    0
SG    0
Name: geoip.region_code, dtype: int64

=== CITY PER COUNTRY ===
geoip.country_code2
US    9
CA    1
SG    1
CH    1
FR    1
CN    1
TH    1
GB    1
BG    0
DE    0
Name: geoip.city_name, dtype: int64

=== UNIQUE COORDINATE PAIRS ===
Unique latitude-longitude pairs: 22
    geoip.latitude  geoip.longitude
34          1.2872         103.8507
22         13.7551         100.5057
4          29.4227         -98.4927
11         32.0589         118.7738
3          33.4532        -112.0748
0          34.0544        -118.2440
1          34.7732         113.7220
13         37.7510         -97.8220
10         38.7095     

In [ ]:
print("=== GEOIP FEATURE CARDINALITY ===\n")

geo_cols = [
    "geoip.country_code2",
    "geoip.region_code",
    "geoip.city_name",
    "geoip.asn",
    "geoip.continent_code",
    "geoip.timezone",
    "geoip.latitude",
    "geoip.longitude"
]

for col in geo_cols:
    print(
        f"{col:30} | "
        f"unique={ftp_clean[col].nunique(dropna=True):2} | "
        f"missing={ftp_clean[col].isna().sum():2}"
    )

=== GEOIP FEATURE CARDINALITY ===

geoip.country_code2            | unique=10 | missing= 0
geoip.region_code              | unique=14 | missing=32
geoip.city_name                | unique=16 | missing=35
geoip.asn                      | unique=20 | missing= 0
geoip.continent_code           | unique= 3 | missing= 0
geoip.timezone                 | unique=13 | missing= 0
geoip.latitude                 | unique=22 | missing= 0
geoip.longitude                | unique=22 | missing= 0


In [ ]:
geoip_drop = [
    "geoip.continent_code",
    "geoip.timezone",
    "geoip.latitude",
    "geoip.longitude"
]

ftp_clean = ftp_clean.drop(columns=geoip_drop)

print("Dropped GeoIP columns:")
print(geoip_drop)

print("\nNew shape:", ftp_clean.shape)

Dropped GeoIP columns:
['geoip.continent_code', 'geoip.timezone', 'geoip.latitude', 'geoip.longitude']

New shape: (76, 46)


In [ ]:
print("geoip.asn:", ftp_clean["geoip.asn"].nunique())
print("geoip.as_org:", ftp_clean["geoip.as_org"].nunique())

print(
    ftp_clean[["geoip.asn", "geoip.as_org"]]
    .drop_duplicates()
    .sort_values("geoip.asn")
)

geoip.asn: 20
geoip.as_org: 20
    geoip.asn                              geoip.as_org
11       4134                                  Chinanet
1        4808     China Unicom Beijing Province Network
21       4837            CHINA UNICOM China169 Backbone
13       6939                                 HURRICANE
3        8075               MICROSOFT-CORP-MSN-AS-BLOCK
55      10439                                   CARINET
57      14061                          DIGITALOCEAN-ASN
30      14618                                AMAZON-AES
14      15169                                    GOOGLE
68      16276                                   OVH SAS
58      48090                       Techoff Srv Limited
50      63949                    Akamai Connected Cloud
22     135377  UCLOUD INFORMATION TECHNOLOGY HK LIMITED
74     136180         Beijing Tiantexin Tech. Co., Ltd.
47     211298                              Driftnet Ltd
12     212512     Detai Prosperous Technologies Limited
19     213412    

In [ ]:
# Remove redundant AS Organization
ftp_clean = ftp_clean.drop(columns=["geoip.as_org"])

print("Dropped:", ["geoip.as_org"])
print("New shape:", ftp_clean.shape)

Dropped: ['geoip.as_org']
New shape: (76, 45)


In [ ]:
print("=== FINAL GEOIP REDUNDANCY AUDIT ===\n")

geo_cols = [
    "geoip.ip",
    "geoip.country_code2",
    "geoip.country_name",
    "geoip.region_code",
    "geoip.region_name",
    "geoip.city_name",
    "geoip.asn",
    "geoip.postal_code",
    "geoip.dma_code"
]

for col in geo_cols:
    print(
        f"{col:30} | "
        f"missing={ftp_clean[col].isna().sum():2} | "
        f"unique={ftp_clean[col].nunique(dropna=True):2}"
    )

=== FINAL GEOIP REDUNDANCY AUDIT ===

geoip.ip                       | missing= 0 | unique=53
geoip.country_code2            | missing= 0 | unique=10
geoip.country_name             | missing= 0 | unique=10
geoip.region_code              | missing=32 | unique=14
geoip.region_name              | missing=32 | unique=14
geoip.city_name                | missing=35 | unique=16
geoip.asn                      | missing= 0 | unique=20
geoip.postal_code              | missing=40 | unique=14
geoip.dma_code                 | missing=55 | unique= 7


In [ ]:
print("\n=== GEOIP IP REDUNDANCY ===")

print(
    "geoip.ip == src_ip:",
    ftp_clean["geoip.ip"].equals(ftp_clean["src_ip"])
)

print("\n=== COUNTRY REDUNDANCY ===")

print(
    ftp_clean[
        ["geoip.country_name", "geoip.country_code2"]
    ].drop_duplicates()
)

print("\n=== REGION REDUNDANCY ===")

print(
    ftp_clean[
        ["geoip.region_code", "geoip.region_name"]
    ].drop_duplicates()
)

print("\n=== POSTAL / DMA COVERAGE ===")

print(
    "Postal coverage:",
    ftp_clean["geoip.postal_code"].notna().sum(),
    "/",
    len(ftp_clean)
)

print(
    "DMA coverage:",
    ftp_clean["geoip.dma_code"].notna().sum(),
    "/",
    len(ftp_clean)
)


=== GEOIP IP REDUNDANCY ===
geoip.ip == src_ip: True

=== COUNTRY REDUNDANCY ===
   geoip.country_name geoip.country_code2
0       United States                  US
1               China                  CN
12            Germany                  DE
14        Switzerland                  CH
19             France                  FR
22           Thailand                  TH
34          Singapore                  SG
47     United Kingdom                  GB
58           Bulgaria                  BG
64             Canada                  CA

=== REGION REDUNDANCY ===
   geoip.region_code geoip.region_name
0                 CA        California
1                NaN               NaN
3                 AZ           Arizona
4                 TX             Texas
10                VA          Virginia
11                JS           Jiangsu
14                ZH            Zurich
19                59             North
22                10           Bangkok
32                IA              Iowa


In [ ]:
geo_drop = [
    "geoip.ip",
    "geoip.country_name",
    "geoip.region_name"
]

ftp_clean = ftp_clean.drop(columns=geo_drop)

print("Dropped GeoIP columns:")
print(geo_drop)

print("\nNew shape:", ftp_clean.shape)

Dropped GeoIP columns:
['geoip.ip', 'geoip.country_name', 'geoip.region_name']

New shape: (76, 42)


In [ ]:
print("=== REMAINING GEOIP FEATURES ===\n")

geo_cols = [
    col for col in ftp_clean.columns
    if col.startswith("geoip.")
]

for col in geo_cols:
    print(
        f"{col:30} | "
        f"missing={ftp_clean[col].isna().sum():2} | "
        f"unique={ftp_clean[col].nunique(dropna=True):2}"
    )

=== REMAINING GEOIP FEATURES ===

geoip.country_code2            | missing= 0 | unique=10
geoip.country_code3            | missing= 0 | unique=10
geoip.region_code              | missing=32 | unique=14
geoip.location.lat             | missing= 0 | unique=22
geoip.location.lon             | missing= 0 | unique=22
geoip.asn                      | missing= 0 | unique=20
geoip.postal_code              | missing=40 | unique=14
geoip.dma_code                 | missing=55 | unique= 7
geoip.city_name                | missing=35 | unique=16


In [ ]:
geo_drop_final = [
    "geoip.country_code3",
    "geoip.location.lat",
    "geoip.location.lon",
    "geoip.postal_code",
    "geoip.dma_code"
]

ftp_clean = ftp_clean.drop(columns=geo_drop_final)

print("Dropped GeoIP columns:")
for col in geo_drop_final:
    print("-", col)

print("\nNew shape:", ftp_clean.shape)

Dropped GeoIP columns:
- geoip.country_code3
- geoip.location.lat
- geoip.location.lon
- geoip.postal_code
- geoip.dma_code

New shape: (76, 37)


In [ ]:
print("=== NETWORK FEATURES AUDIT ===\n")

network_cols = [
    "src_ip",
    "src_port",
    "dest_ip",
    "t-pot_ip_int",
    "t-pot_ip_ext",
    "host"
]

for col in network_cols:
    print(
        f"{col:20} | "
        f"missing={ftp_clean[col].isna().sum():2} | "
        f"unique={ftp_clean[col].nunique(dropna=True):2}"
    )

=== NETWORK FEATURES AUDIT ===

src_ip               | missing= 0 | unique=53
src_port             | missing= 0 | unique=75
dest_ip              | missing= 0 | unique= 4
t-pot_ip_int         | missing= 0 | unique= 4
t-pot_ip_ext         | missing= 0 | unique=10
host                 | missing= 0 | unique= 2


In [ ]:
print("=== NETWORK REDUNDANCY CHECK ===\n")

print("dest_ip == t-pot_ip_int:",
      ftp_clean["dest_ip"].equals(ftp_clean["t-pot_ip_int"]))

print("\n=== DESTINATION IP ===")
print(ftp_clean["dest_ip"].value_counts())

print("\n=== T-POT INTERNAL IP ===")
print(ftp_clean["t-pot_ip_int"].value_counts())

print("\n=== T-POT EXTERNAL IP ===")
print(ftp_clean["t-pot_ip_ext"].value_counts())

print("\n=== HOST ===")
print(ftp_clean["host"].value_counts())

print("\n=== T-POT EXTERNAL IP vs HOST ===")
print(
    ftp_clean[["t-pot_ip_ext", "host"]]
    .drop_duplicates()
    .sort_values(["t-pot_ip_ext", "host"])
)

=== NETWORK REDUNDANCY CHECK ===

dest_ip == t-pot_ip_int: False

=== DESTINATION IP ===
dest_ip
172.19.0.2    30
172.21.0.2    18
172.20.0.2    18
172.18.0.2    10
Name: count, dtype: int64

=== T-POT INTERNAL IP ===
t-pot_ip_int
10.3.0.4    29
10.4.0.4    20
10.1.0.4    16
10.2.0.4    11
Name: count, dtype: int64

=== T-POT EXTERNAL IP ===
t-pot_ip_ext
68.221.197.138    24
20.87.74.66       18
135.235.171.89    14
68.221.134.12      5
52.173.49.247      3
52.185.37.223      3
40.86.6.196        3
4.221.107.209      2
4.247.175.193      2
52.176.6.22        2
Name: count, dtype: int64

=== HOST ===
host
172.18.0.6    48
172.18.0.2    28
Name: count, dtype: int64

=== T-POT EXTERNAL IP vs HOST ===
      t-pot_ip_ext        host
13  135.235.171.89  172.18.0.2
40  135.235.171.89  172.18.0.6
19     20.87.74.66  172.18.0.2
48     20.87.74.66  172.18.0.6
0    4.221.107.209  172.18.0.6
5    4.247.175.193  172.18.0.6
44     40.86.6.196  172.18.0.6
72   52.173.49.247  172.18.0.6
27     52.176.

In [ ]:
print("=== FINAL NETWORK FEATURES ===\n")

network_cols = [
    "src_ip",
    "src_port",
    "dest_ip",
    "t-pot_ip_int",
    "t-pot_ip_ext",
    "host"
]

for col in network_cols:
    print(
        f"{col:20} | "
        f"missing={ftp_clean[col].isna().sum():2} | "
        f"unique={ftp_clean[col].nunique(dropna=True):2}"
    )

=== FINAL NETWORK FEATURES ===

src_ip               | missing= 0 | unique=53
src_port             | missing= 0 | unique=75
dest_ip              | missing= 0 | unique= 4
t-pot_ip_int         | missing= 0 | unique= 4
t-pot_ip_ext         | missing= 0 | unique=10
host                 | missing= 0 | unique= 2


In [ ]:
print("=== FINAL FTP FEATURE AUDIT ===\n")

for i, col in enumerate(ftp_clean.columns, 1):
    print(f"{i:2}. {col}")

print("\nFinal shape:", ftp_clean.shape)

=== FINAL FTP FEATURE AUDIT ===

 1. t-pot_hostname
 2. t-pot_ip_int
 3. t-pot_ip_ext
 4. src_port
 5. @timestamp
 6. host
 7. dest_ip
 8. src_ip
 9. geoip_ext.timezone
10. geoip_ext.latitude
11. geoip_ext.postal_code
12. geoip_ext.country_code2
13. geoip_ext.country_name
14. geoip_ext.country_code3
15. geoip_ext.region_code
16. geoip_ext.continent_code
17. geoip_ext.region_name
18. geoip_ext.city_name
19. geoip_ext.longitude
20. geoip_ext.location.lat
21. geoip_ext.location.lon
22. geoip_ext.ip
23. headers.content_length
24. geoip.country_code2
25. geoip.region_code
26. geoip.asn
27. geoip_ext.dma_code
28. geoip.city_name
29. ip_rep
30. command_count
31. has_USER
32. has_PASS
33. has_AUTH
34. ftp_commands_clean
35. has_username
36. has_password
37. is_anonymous

Final shape: (76, 37)


In [ ]:
ftp_features = [
    "command_count",
    "has_USER",
    "has_PASS",
    "has_AUTH",
    "has_username",
    "has_password",
    "is_anonymous"
]

print("=== FTP FEATURES CHECK ===\n")

for col in ftp_features:
    print(
        f"{col:20} | "
        f"unique={ftp_clean[col].nunique(dropna=True)} | "
        f"missing={ftp_clean[col].isna().sum()}"
    )

=== FTP FEATURES CHECK ===

command_count        | unique=6 | missing=0
has_USER             | unique=2 | missing=0
has_PASS             | unique=2 | missing=0
has_AUTH             | unique=2 | missing=0
has_username         | unique=2 | missing=0
has_password         | unique=2 | missing=0
is_anonymous         | unique=2 | missing=0


In [ ]:
missing_final = (
    ftp_clean.isna()
    .sum()
    .to_frame("missing_count")
)

missing_final["missing_percentage"] = (
    missing_final["missing_count"] / len(ftp_clean) * 100
)

print("=== FINAL MISSING VALUES ===")
print(
    missing_final[
        missing_final["missing_count"] > 0
    ].sort_values("missing_count", ascending=False)
)

=== FINAL MISSING VALUES ===
                    missing_count  missing_percentage
geoip_ext.dma_code             65           85.526316
ip_rep                         38           50.000000
geoip.city_name                35           46.052632
geoip.region_code              32           42.105263


In [ ]:
# Remove intermediate FTP command feature
ftp_final = ftp_clean.drop(columns=["ftp_commands_clean"]).copy()

print("=== FINAL FTP DATASET ===")
print("Shape:", ftp_final.shape)

print("\nColumns:")
for i, col in enumerate(ftp_final.columns, 1):
    print(f"{i:2}. {col}")

print("\nDuplicate rows:", ftp_final.duplicated().sum())

=== FINAL FTP DATASET ===
Shape: (76, 36)

Columns:
 1. t-pot_hostname
 2. t-pot_ip_int
 3. t-pot_ip_ext
 4. src_port
 5. @timestamp
 6. host
 7. dest_ip
 8. src_ip
 9. geoip_ext.timezone
10. geoip_ext.latitude
11. geoip_ext.postal_code
12. geoip_ext.country_code2
13. geoip_ext.country_name
14. geoip_ext.country_code3
15. geoip_ext.region_code
16. geoip_ext.continent_code
17. geoip_ext.region_name
18. geoip_ext.city_name
19. geoip_ext.longitude
20. geoip_ext.location.lat
21. geoip_ext.location.lon
22. geoip_ext.ip
23. headers.content_length
24. geoip.country_code2
25. geoip.region_code
26. geoip.asn
27. geoip_ext.dma_code
28. geoip.city_name
29. ip_rep
30. command_count
31. has_USER
32. has_PASS
33. has_AUTH
34. has_username
35. has_password
36. is_anonymous

Duplicate rows: 0


In [ ]:
ftp_final.to_csv(
    "/content/ftp_clean_final.csv",
    index=False
)

print("FTP CSV saved successfully!")

FTP CSV saved successfully!
